In [ ]:
# 1. Libraries
import pandas as pd
import numpy as np
import os
import warnings
from sklearn.model_selection import KFold
warnings.filterwarnings('ignore')

In [ ]:
# 2. CSV를 Parquet으로 변환 (메모리 효율화)
csv_dir = "./data/raw"
parquet_dir = "./data/processed"

os.makedirs(parquet_dir, exist_ok=True)

file_mapping = {
    'train.csv': 'train.parquet',
    'test.csv': 'test.parquet',
    'sample_submission.csv': 'submission.parquet'
}

for csv_name, parquet_name in file_mapping.items():
    df = pd.read_csv(f"{csv_dir}/{csv_name}", low_memory=False)
    df.to_parquet(
        f"{parquet_dir}/{parquet_name}",
        compression='snappy',
        index=False
    )
    print(f"✓ {csv_name}: {df.shape}")
    del df

# 3. Parquet 파일 로딩
train = pd.read_parquet(f"{parquet_dir}/train.parquet")
test = pd.read_parquet(f"{parquet_dir}/test.parquet")

✓ train.csv: (28605391, 41)
✓ test.csv: (4538541, 40)
✓ sample_submission.csv: (4538541, 2)


In [ ]:
print(f"Initial memory:")
train_initial = train.memory_usage(deep=True).sum() / 1024**3
test_initial = test.memory_usage(deep=True).sum() / 1024**3
print(f"Train: {train_initial:.2f} GB")
print(f"Test: {test_initial:.2f} GB")

# 4. ID Processing
if 'ID' in train.columns and train['ID'].dtype == 'object':
    try:
        train['ID'] = train['ID'].str.extract(r'(\d+)')[0].astype('int32')
        test['ID'] = test['ID'].str.extract(r'(\d+)')[0].astype('int32')
        print(f"\nID processed: object → int32")
    except:
        pass

# 5. Category Conversion (except ID)
def convert_to_category(df):
    """Object to Category conversion"""
    for col in df.select_dtypes(include=['object']).columns:
        if col != 'ID':
            df[col] = df[col].astype('category')
    return df

train = convert_to_category(train)
test = convert_to_category(test)

print(f"\nAfter category conversion:")
train_after_category = train.memory_usage(deep=True).sum() / 1024**3
test_after_category = test.memory_usage(deep=True).sum() / 1024**3
print(f"Train: {train_after_category:.2f} GB")
print(f"Test: {test_after_category:.2f} GB")

category_reduction_train = (1 - train_after_category/train_initial) * 100
category_reduction_test = (1 - test_after_category/test_initial) * 100
print(f"Reduction - Train: {category_reduction_train:.1f}%, Test: {category_reduction_test:.1f}%")

# 6. Downcasting
def downcast_dtypes(df):
    """Numeric downcasting"""
    for col in df.columns:
        if df[col].dtype == 'float64':
            df[col] = df[col].astype('float32')
        elif df[col].dtype == 'int64':
            df[col] = pd.to_numeric(df[col], downcast='integer')
    return df

train = downcast_dtypes(train)
test = downcast_dtypes(test)

print(f"\nAfter downcasting:")
train_after_downcast = train.memory_usage(deep=True).sum() / 1024**3
test_after_downcast = test.memory_usage(deep=True).sum() / 1024**3
print(f"Train: {train_after_downcast:.2f} GB")
print(f"Test: {test_after_downcast:.2f} GB")

downcast_reduction_train = (1 - train_after_downcast/train_after_category) * 100
downcast_reduction_test = (1 - test_after_downcast/test_after_category) * 100
print(f"Reduction - Train: {downcast_reduction_train:.1f}%, Test: {downcast_reduction_test:.1f}%")

total_reduction_train = (1 - train_after_downcast/train_initial) * 100
total_reduction_test = (1 - test_after_downcast/test_initial) * 100
print(f"\nTotal reduction:")
print(f"Train: {train_initial:.2f} GB → {train_after_downcast:.2f} GB ({total_reduction_train:.1f}%)")
print(f"Test: {test_initial:.2f} GB → {test_after_downcast:.2f} GB ({total_reduction_test:.1f}%)")

# 7. Target Encoding
def kfold_target_encoding(train, test, cat_cols, target_col, n_splits=5, smoothing=10):
    """K-Fold Target Encoding with Smoothing"""
    global_mean = train[target_col].mean()
    train_encoded = pd.DataFrame(index=train.index)
    test_encoded = pd.DataFrame(index=test.index)

    for col in cat_cols:
        train_col_str = train[col].astype(str)
        test_col_str = test[col].astype(str)
        train_encoded[col] = np.nan

        kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

        for train_idx, val_idx in kf.split(train):
            fold_train = pd.DataFrame({
                'cat': train_col_str.iloc[train_idx],
                'target': train[target_col].iloc[train_idx]
            })
            stats = fold_train.groupby('cat')['target'].agg(['mean', 'count'])
            smoothed = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
            smoothed_dict = smoothed.to_dict()

            val_categories = train_col_str.iloc[val_idx]
            encoded_values = val_categories.map(smoothed_dict).values
            train_encoded.iloc[val_idx, train_encoded.columns.get_loc(col)] = encoded_values

        full_train = pd.DataFrame({
            'cat': train_col_str,
            'target': train[target_col]
        })
        stats = full_train.groupby('cat')['target'].agg(['mean', 'count'])
        smoothed = (stats['count'] * stats['mean'] + smoothing * global_mean) / (stats['count'] + smoothing)
        smoothed_dict = smoothed.to_dict()

        test_encoded[col] = test_col_str.map(smoothed_dict).fillna(global_mean)

        train_encoded[col] = train_encoded[col].astype('float32')
        test_encoded[col] = test_encoded[col].astype('float32')

    return train_encoded, test_encoded

cat_cols = [col for col in train.select_dtypes(include=['category']).columns
            if col not in ['Click', 'ID']]

train_te, test_te = kfold_target_encoding(train, test, cat_cols, 'Click', n_splits=5, smoothing=10)

train_final = train.drop(columns=cat_cols).join(train_te)
test_final = test.drop(columns=cat_cols).join(test_te)

Initial memory:
Train: 41.97 GB
Test: 6.63 GB

ID processed: object → int32

After category conversion:
Train: 6.20 GB
Test: 1.06 GB
Reduction - Train: 85.2%, Test: 84.0%

After downcasting:
Train: 4.58 GB
Test: 0.83 GB
Reduction - Train: 26.2%, Test: 22.3%

Total reduction:
Train: 41.97 GB → 4.58 GB (89.1%)
Test: 6.63 GB → 0.83 GB (87.5%)


In [ ]:


# 8. Prepare for Modeling
import time
import gc
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# X, y split
X = train_final.drop(['ID', 'Click'], axis=1)
y = train_final['Click']

# Test data
X_test = test_final.drop(['ID'], axis=1)
test_ids = test_final['ID']

print(f"Train shape: {X.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Click rate: {y.mean():.4f}")

# Train/Valid split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Click rate: {y_train.mean():.4f}")
print(f"Valid: {X_valid.shape}, Click rate: {y_valid.mean():.4f}")

# Memory cleanup
del X, y, train, test, train_te, test_te
gc.collect()

# 9. XGBoost Training
start_xgb = time.time()

xgb_params = {
    'n_estimators': 1000,
    'learning_rate': 0.05,
    'max_depth': 8,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1,
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'device': 'cuda',
    'random_state': 42
}

xgb_model = XGBClassifier(**xgb_params)
xgb_model.fit(X_train, y_train, eval_set=[(X_valid, y_valid)], verbose=100)

xgb_valid_pred = xgb_model.predict_proba(X_valid)[:, 1]
xgb_auc = roc_auc_score(y_valid, xgb_valid_pred)
xgb_time = time.time() - start_xgb

print(f"\nXGBoost AUC: {xgb_auc:.4f} | Time: {xgb_time/60:.1f}min")

Train shape: (28605391, 39)
Test shape: (4538541, 39)
Click rate: 0.1947
Train: (22884312, 39), Click rate: 0.1947
Valid: (5721079, 39), Click rate: 0.1947
[0]	validation_0-auc:0.73679
[100]	validation_0-auc:0.76975
[200]	validation_0-auc:0.77428
[300]	validation_0-auc:0.77652
[400]	validation_0-auc:0.77799
[500]	validation_0-auc:0.77899
[600]	validation_0-auc:0.77987
[700]	validation_0-auc:0.78054
[800]	validation_0-auc:0.78113
[900]	validation_0-auc:0.78158
[999]	validation_0-auc:0.78198

XGBoost AUC: 0.7820 | Time: 1.7min


In [ ]:
# 10. Test Prediction & Submission
print("\nTest prediction...")

# Feature 순서 맞추기
train_features = X_train.columns.tolist()
X_test_ordered = X_test[train_features]

# 예측
xgb_test_pred = xgb_model.predict_proba(X_test_ordered)[:, 1]
print(f"✅ 예측 완료 (예측값 평균: {xgb_test_pred.mean():.6f})")




Test prediction...
✅ 예측 완료 (예측값 평균: 0.195903)


In [ ]:
# 제출 파일 생성 및 저장
submission = pd.DataFrame({
    'ID': ['TEST_' + str(i).zfill(7) for i in range(len(xgb_test_pred))],
    'Click': xgb_test_pred
})

print("\n첫 10개:")
print(submission.head(10))

submission_path = "./submission_xgb.csv"
submission.to_csv(submission_path, index=False)
print(f"✅ 저장 완료: {submission_path}")


첫 10개:
             ID     Click
0  TEST_0000000  0.216441
1  TEST_0000001  0.148838
2  TEST_0000002  0.096675
3  TEST_0000003  0.361375
4  TEST_0000004  0.366469
5  TEST_0000005  0.101323
6  TEST_0000006  0.286908
7  TEST_0000007  0.026011
8  TEST_0000008  0.285367
9  TEST_0000009  0.061425
✅ 저장 완료: ./submission_xgb.csv
